In [ ]:
!pip install -q "transformers>=4.40,<5.0" accelerate torchaudio librosa scikit-learn
!apt-get -y -q install ffmpeg > /dev/null 2>&1

!git clone -q https://github.com/m3hrdadfi/soxan.git
import sys
sys.path.append("/content/soxan")
from src.models import Wav2Vec2ForSpeechClassification

import torch
print("GPU:", torch.cuda.is_available())
from google.colab import drive
drive.mount('/content/drive')

fatal: destination path 'soxan' already exists and is not an empty directory.
GPU: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import AutoConfig, Wav2Vec2FeatureExtractor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PHASE1_CHECKPOINT = "/content/drive/MyDrive/final_project/shemo_phase1_checkpoint"

config_p1 = AutoConfig.from_pretrained(PHASE1_CHECKPOINT)
feature_extractor_p1 = Wav2Vec2FeatureExtractor.from_pretrained(PHASE1_CHECKPOINT)
target_sr_p1 = feature_extractor_p1.sampling_rate

phase1_model = Wav2Vec2ForSpeechClassification.from_pretrained(PHASE1_CHECKPOINT).to(device)
phase1_model.eval()

print("Phase 1 model labels:", config_p1.id2label)

Phase 1 model labels: {0: 'anger', 1: 'happiness', 2: 'neutral', 3: 'sadness'}


In [ ]:
import glob, os, subprocess, shutil, time
from collections import Counter

TEST_DATA_DIR = "/content/drive/MyDrive/final_project/test/audio"
CONVERTED_DIR = "/content/test_audio_wav"
TMP_SRC_DIR = "/content/tmp_src"

AUDIO_EXTENSIONS = ["m4a", "mp3", "wav", "flac", "ogg", "aac", "wma"]

audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(glob.glob(f"{TEST_DATA_DIR}/**/*.{ext}", recursive=True))
    audio_files.extend(glob.glob(f"{TEST_DATA_DIR}/**/*.{ext.upper()}", recursive=True))

print(f"Number of audio files found: {len(audio_files)}")

os.makedirs(CONVERTED_DIR, exist_ok=True)
os.makedirs(TMP_SRC_DIR, exist_ok=True)

MIN_VALID_SIZE = 1000

def safe_convert(src, out_path, attempts=3):
    if os.path.exists(out_path) and os.path.getsize(out_path) > MIN_VALID_SIZE:
        return True

    for attempt in range(attempts):
        tmp_src = os.path.join(TMP_SRC_DIR, os.path.basename(src))
        shutil.copyfile(src, tmp_src)

        result = subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_src, "-ar", "16000", "-ac", "1", out_path],
            stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
        )
        os.remove(tmp_src)

        ok = result.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > MIN_VALID_SIZE
        if ok:
            return True
        if os.path.exists(out_path):
            os.remove(out_path)
        time.sleep(1)

    return False

failed_conversions = []
for f in audio_files:
    speaker = os.path.basename(os.path.dirname(f))
    out_dir = os.path.join(CONVERTED_DIR, speaker)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + ".wav")
    if not safe_convert(f, out_path):
        failed_conversions.append(f)

if failed_conversions:
    print(f"{len(failed_conversions)} files could not be converted even after retry (source file is likely corrupted):")
    for f in failed_conversions:
        size = os.path.getsize(f) if os.path.exists(f) else "not found"
        print(f"  {f}  (source size: {size})")
else:
    print("All files were successfully converted.")

Number of audio files found: 93
All files were successfully converted.


In [ ]:
LABEL_CODE_MAP = {"ANG": "anger", "HAP": "happiness", "NEU": "neutral", "SAD": "sadness"}
def get_label_from_filename(fp):
    name = os.path.splitext(os.path.basename(fp))[0].upper()
    for code, emo in LABEL_CODE_MAP.items():
        if code in name:
            return emo
    return None

test_speaker_dirs = sorted(glob.glob(f"{CONVERTED_DIR}/speaker_*"))
test_data = []  # (path, true_label)
for d in test_speaker_dirs:
    for f in glob.glob(f"{d}/*.wav"):
        if os.path.getsize(f) <= MIN_VALID_SIZE:
            continue
        lb = get_label_from_filename(f)
        if lb:
            test_data.append((f, lb))

print(f"Number of speakers: {len(test_speaker_dirs)} | Number of samples: {len(test_data)}")
print("Class distribution:", Counter([l for _, l in test_data]))

Number of speakers: 22 | Number of samples: 93
Class distribution: Counter({'anger': 25, 'neutral': 23, 'happiness': 23, 'sadness': 22})


In [ ]:
import librosa

def speech_file_to_array_fn(path, target_sr):
    speech_array, _ = librosa.load(path, sr=target_sr, mono=True)
    return speech_array

@torch.no_grad()
def predict_phase1(path):
    speech = speech_file_to_array_fn(path, target_sr_p1)
    inputs = feature_extractor_p1(speech, sampling_rate=target_sr_p1, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)
    logits = phase1_model(input_values).logits
    pred_id = torch.argmax(logits, dim=-1).item()
    return config_p1.id2label[pred_id].lower()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

y_true_p1, y_pred_p1 = [], []
for path, true_label in test_data:
    pred_label = predict_phase1(path)
    y_true_p1.append(true_label)
    y_pred_p1.append(pred_label)

print(f"Model accuracy after data augmentation (Phase 1) on held-out data: {accuracy_score(y_true_p1, y_pred_p1):.4f}")
print(classification_report(y_true_p1, y_pred_p1))

labels_order = sorted(set(y_true_p1) | set(y_pred_p1))
cm = confusion_matrix(y_true_p1, y_pred_p1, labels=labels_order)
pd.DataFrame(cm, index=labels_order, columns=labels_order)

Model accuracy after data augmentation (Phase 1) on held-out data: 0.4516
              precision    recall  f1-score   support

       anger       1.00      0.24      0.39        25
   happiness       0.50      0.22      0.30        23
     neutral       0.37      0.65      0.47        23
     sadness       0.44      0.73      0.55        22

    accuracy                           0.45        93
   macro avg       0.58      0.46      0.43        93
weighted avg       0.59      0.45      0.43        93



,anger,happiness,neutral,sadness
anger,6,2,16,1
happiness,0,5,6,12
neutral,0,1,15,7
sadness,0,2,4,16
